# MLflow notebook: NYC Green Taxi fare regression

**Tasks to do:**

1. Create 3 experiments for 3 different regression models on the data (green taxi 
January and February 2021 combined) that we used in previous practices to train 
models. Make different runs in each experiment to get the best run. Explain 
everything in the report with screenshots.
2. Test your best models on the March 2021 data (uploaded with this practice) and 
assign them stages accordingly. Test the models and put best one in production 
keep other two in staging. Do it in code not on the MLflow UI.
3. Reproduce your best model from Question 1 using MLflow and verify that you get 
the same results
    * a. Load the model from MLflow using its run ID or model URI
    * b. Run inference again on the same test dataset
    * c. Compare the predictions and evaluation metrics
    * d. Explain whether the results match exactly or not and why

## 0. Imports and basic setup

A few practical notes:

- I use a **fixed random seed** so results are easier to reproduce.
- The features are kept intentionally close to the regression example used previously.

In [ ]:
from pathlib import Path
import itertools
import warnings

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

warnings.filterwarnings("ignore")

# Change this if your MLflow server is elsewhere.
# If you already run "mlflow server ..." locally, this is a common default.
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

RANDOM_STATE = 42

# Notebook is assumed to live in dev/, and data/ is next to it.
DATA_DIR = Path("../data")

JAN_PATH = DATA_DIR / "green_tripdata_2021-01.parquet"
FEB_PATH = DATA_DIR / "green_tripdata_2021-02.parquet"
MAR_PATH = DATA_DIR / "green_tripdata_2021-03.parquet"

FEATURES = [
    "trip_distance",
    "trip_duration_min",
    "passenger_count",
    "RatecodeID",
    "pickup_hour",
]
TARGET = "fare_amount"

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Expected data files:")
print(" -", JAN_PATH)
print(" -", FEB_PATH)
print(" -", MAR_PATH)

MLflow tracking URI: http://127.0.0.1:5000
Expected data files:
 - ../data/green_tripdata_2021-01.parquet
 - ../data/green_tripdata_2021-02.parquet
 - ../data/green_tripdata_2021-03.parquet


## 1. Helper functions

In [8]:
def load_parquet_files(paths):
    dfs = [pd.read_parquet(path) for path in paths]
    return pd.concat(dfs, ignore_index=True)


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["lpep_pickup_datetime"] = pd.to_datetime(
        df["lpep_pickup_datetime"], errors="coerce"
    )
    df["lpep_dropoff_datetime"] = pd.to_datetime(
        df["lpep_dropoff_datetime"], errors="coerce"
    )

    # Trip duration in minutes
    df["trip_duration_min"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60.0

    # Simple time-based feature
    df["pickup_hour"] = df["lpep_pickup_datetime"].dt.hour

    return df


def basic_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    needed = set(FEATURES + [TARGET])
    missing_cols = needed - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing expected columns: {sorted(missing_cols)}")

    # Drop rows where critical values are missing
    df = df.dropna(subset=[TARGET, "trip_distance", "trip_duration_min"])

    # Keep only pretty sensible taxi rows
    df = df[df[TARGET].between(0.5, 300)]
    df = df[df["trip_distance"].between(0.01, 100)]
    df = df[df["trip_duration_min"].between(0.1, 300)]

    if "passenger_count" in df.columns:
        df = df[df["passenger_count"].between(1, 8)]

    if "RatecodeID" in df.columns:
        df = df[df["RatecodeID"].between(1, 6)]

    if "pickup_hour" in df.columns:
        df = df[df["pickup_hour"].between(0, 23)]

    return df


def prepare_features_and_target(df: pd.DataFrame):
    X = df[FEATURES].astype(float)
    y = df[TARGET].astype(float)
    return X, y


def evaluate_regression(model, X, y):
    
    preds = model.predict(X)

    mae = mean_absolute_error(y, preds)
    rmse = root_mean_squared_error(y, preds)
    r2 = r2_score(y, preds)

    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    }, preds


def build_pipeline(model):
    # Very simple preprocessing: just fill missing numeric values
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model),
        ]
    )

## 2. Load January + February data and make one train/validation split

In [3]:
train_df = load_parquet_files([JAN_PATH, FEB_PATH])
train_df = add_derived_features(train_df)
train_df = basic_clean(train_df)

X_all, y_all = prepare_features_and_target(train_df)

# This validation split is what we will use to compare runs fairly.
# We keep the random_state fixed so we can reproduce it later.
X_train, X_valid, y_train, y_valid = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("Combined Jan+Feb rows after cleaning:", len(train_df))
print("Train rows:", len(X_train))
print("Validation rows:", len(X_valid))

Combined Jan+Feb rows after cleaning: 71475
Train rows: 57180
Validation rows: 14295


## 3. Define the 3 model families and a few parameter combinations

3 basic model families to be tested:

- **Linear Regression**
- **Random Forest Regressor**
- **Gradient Boosting Regressor**

Each family gets its own MLflow experiment, and each parameter combination becomes one run.

In [20]:
EXPERIMENT_CONFIGS = {
    "nyc-green-taxi-linear-regression": {
        "model_label": "LinearRegression",
        "builder": lambda params: LinearRegression(**params),
        "param_grid": [
            {"fit_intercept": True},
        ],
    },
    "nyc-green-taxi-random-forest": {
        "model_label": "RandomForestRegressor",
        "builder": lambda params: RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params,
        ),
        "param_grid": [
            {"n_estimators": 200, "max_depth": 15, "min_samples_leaf": 2},
        ],
    },
    "nyc-green-taxi-gradient-boosting": {
        "model_label": "GradientBoostingRegressor",
        "builder": lambda params: GradientBoostingRegressor(
            random_state=RANDOM_STATE,
            **params,
        ),
        "param_grid": [
            {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3},
        ],
    },
}

## 4. Train runs and log everything to MLflow

In [21]:
from mlflow.models import infer_signature

mlflow.sklearn.autolog(log_models=False) # Used to be True, added a lot of noisy metrics

best_runs_q1 = []

for experiment_name, config in EXPERIMENT_CONFIGS.items():
    mlflow.set_experiment(experiment_name)
    experiment = mlflow.get_experiment_by_name(experiment_name)

    print(f"\n=== Experiment: {experiment_name} ===")

    # We only have one param set now, but keeping the loop makes the code reusable
    run_results = []

    # Count existing runs so the new run name stays readable in MLflow
    existing_runs_df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        output_format="pandas",
    )
    existing_run_count = 0 if existing_runs_df.empty else len(existing_runs_df)

    for run_offset, params in enumerate(config["param_grid"], start=1):
        run_number = existing_run_count + run_offset
        model = build_pipeline(config["builder"](params))

        with mlflow.start_run(run_name=f"run_{run_number}") as run:
            # Train on the full Jan+Feb training split
            model.fit(X_train, y_train)

            # Evaluate only on validation data
            valid_metrics, valid_preds = evaluate_regression(model, X_valid, y_valid)

            # Keep params explicit and simple
            mlflow.log_param("model_family", config["model_label"])
            mlflow.log_param("features", ",".join(FEATURES))
            mlflow.log_param("target", TARGET)
            mlflow.log_param("random_state", RANDOM_STATE)
            mlflow.log_param("validation_split", 0.2)

            for key, value in params.items():
                mlflow.log_param(key, value)

            # Only log the validation metrics we care about for Q1 comparison
            mlflow.log_metric("valid_mae", valid_metrics["mae"])
            mlflow.log_metric("valid_rmse", valid_metrics["rmse"])
            mlflow.log_metric("valid_r2", valid_metrics["r2"])

            # Add model signature + small input example
            sample = X_train.head(100) # Not the whole dataset is needed for this (slow, unnecessary, wasteful)
            signature = infer_signature(
                sample,
                model.predict(sample) # Just a schema, correct dimensions
                ) 

            mlflow.sklearn.log_model(
                sk_model=model,
                name="model",
                signature=signature,
                input_example=X_train.head(5),
            )

            mlflow.set_tags(
                {
                    "dataset": "NYC green taxi Jan+Feb 2021",
                    "task": "fare_amount regression",
                    "phase": "Q1 model selection",
                    "run_note": "clean rerun on full train/valid split",
                }
            )

            result = {
                "experiment_name": experiment_name,
                "experiment_id": experiment.experiment_id,
                "run_id": run.info.run_id,
                "model_family": config["model_label"],
                "params": params,
                "valid_mae": valid_metrics["mae"],
                "valid_rmse": valid_metrics["rmse"],
                "valid_r2": valid_metrics["r2"],
                "model_uri": f"runs:/{run.info.run_id}/model",
            }
            run_results.append(result)

            print(
                f"run_{run_number} | params={params} | "
                f"valid_rmse={valid_metrics['rmse']:.4f} | "
                f"valid_r2={valid_metrics['r2']:.4f}"
            )

    # Pick the best run across ALL runs in this experiment, using validation RMSE
    all_runs_df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        output_format="pandas",
    )

    all_runs_df = all_runs_df.dropna(subset=["metrics.valid_rmse"])
    best_row = all_runs_df.sort_values("metrics.valid_rmse", ascending=True).iloc[0]

    best_run = {
        "experiment_name": experiment_name,
        "experiment_id": experiment.experiment_id,
        "run_id": best_row["run_id"],
        "model_family": config["model_label"],
        "valid_rmse": best_row["metrics.valid_rmse"],
        "valid_r2": best_row.get("metrics.valid_r2"),
        "model_uri": f"runs:/{best_row['run_id']}/model",
    }
    best_runs_q1.append(best_run)

    print("Best run in this experiment so far:")
    print(best_run)


=== Experiment: nyc-green-taxi-linear-regression ===
run_4 | params={'fit_intercept': True} | valid_rmse=3.3951 | valid_r2=0.9426
🏃 View run run_4 at: http://127.0.0.1:5000/#/experiments/1/runs/468d876f0432457a95c507c89f4d4791
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Best run in this experiment so far:
{'experiment_name': 'nyc-green-taxi-linear-regression', 'experiment_id': '1', 'run_id': '468d876f0432457a95c507c89f4d4791', 'model_family': 'LinearRegression', 'valid_rmse': np.float64(3.3951356953034812), 'valid_r2': np.float64(0.9425661206031403), 'model_uri': 'runs:/468d876f0432457a95c507c89f4d4791/model'}

=== Experiment: nyc-green-taxi-random-forest ===
run_5 | params={'n_estimators': 200, 'max_depth': 15, 'min_samples_leaf': 2} | valid_rmse=2.7827 | valid_r2=0.9614
🏃 View run run_5 at: http://127.0.0.1:5000/#/experiments/2/runs/a08dac2cc0cf4648ae09fbaef90f0bfb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Best run in this experiment so far:
{'exper

## 5. Best run from each experiment (Question 1 result)

In [22]:
best_runs_q1_df = pd.DataFrame(best_runs_q1).sort_values("valid_rmse").reset_index(drop=True)
best_runs_q1_df

,experiment_name,experiment_id,run_id,model_family,valid_rmse,valid_r2,model_uri
0,nyc-green-taxi-gradient-boosting,3,9da3c8600a694c50a00afea247b79b8b,GradientBoostingRegressor,2.614181,0.965949,runs:/9da3c8600a694c50a00afea247b79b8b/model
1,nyc-green-taxi-random-forest,2,829e3963138a48ec9f37cd702ee7afac,RandomForestRegressor,2.710555,0.963392,runs:/829e3963138a48ec9f37cd702ee7afac/model
2,nyc-green-taxi-linear-regression,1,468d876f0432457a95c507c89f4d4791,LinearRegression,3.395136,0.942566,runs:/468d876f0432457a95c507c89f4d4791/model


The **overall best model from Question 1** is just the one with the lowest validation RMSE among those 3 experiment winners.

In [23]:
overall_best_q1 = best_runs_q1_df.iloc[0].to_dict()
overall_best_q1

{'experiment_name': 'nyc-green-taxi-gradient-boosting',
 'experiment_id': '3',
 'run_id': '9da3c8600a694c50a00afea247b79b8b',
 'model_family': 'GradientBoostingRegressor',
 'valid_rmse': 2.6141808076885003,
 'valid_r2': 0.9659493722886188,
 'model_uri': 'runs:/9da3c8600a694c50a00afea247b79b8b/model'}

## 6. Load March 2021 data for final testing (Question 2)

In [26]:
march_df = pd.read_parquet(MAR_PATH)
march_df = add_derived_features(march_df)
march_df = basic_clean(march_df)

X_march, y_march = prepare_features_and_target(march_df)

print("March rows after cleaning:", len(march_df))

March rows after cleaning: 40677


## 7. Test the best model from each experiment on March and register them in MLflow

Steps:

1. load each winning model from its **run ID / model URI**
2. evaluate it on the **March 2021** dataset
3. register it in the MLflow **Model Registry**
4. assign stages **in code**

In [29]:
client = MlflowClient()

registered_results = []

for row in best_runs_q1:
    model_uri = row["model_uri"]
    loaded_model = mlflow.sklearn.load_model(model_uri)

    # Evaluate on March
    test_metrics, _ = evaluate_regression(loaded_model, X_march, y_march)

    # Small readable registry name
    registry_name = f"{row['experiment_name']}-model"

    # Create the registered model if needed
    try:
        client.create_registered_model(registry_name)
    except Exception:
        pass

    # Register model version from the winning run
    model_version = mlflow.register_model(
        model_uri=model_uri,
        name=registry_name,
    )

    result = {
        **row,
        "march_mae": test_metrics["mae"],
        "march_rmse": test_metrics["rmse"],
        "march_r2": test_metrics["r2"],
        "registry_name": registry_name,
        "model_version": int(model_version.version),
    }
    registered_results.append(result)

registered_results_df = (
    pd.DataFrame(registered_results)
    .sort_values("march_rmse")
    .reset_index(drop=True)
)

registered_results_df

Registered model 'nyc-green-taxi-linear-regression-model' already exists. Creating a new version of this model...
2026/03/24 10:05:18 WARNING mlflow.tracking._model_registry.fluent: Run with id 468d876f0432457a95c507c89f4d4791 has no artifacts at artifact path 'model', registering model based on models:/m-991bdd4b49f5430296dae577ef0c218d instead
2026/03/24 10:05:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-green-taxi-linear-regression-model, version 3
Created version '3' of model 'nyc-green-taxi-linear-regression-model'.
Registered model 'nyc-green-taxi-random-forest-model' already exists. Creating a new version of this model...
2026/03/24 10:05:19 WARNING mlflow.tracking._model_registry.fluent: Run with id 829e3963138a48ec9f37cd702ee7afac has no artifacts at artifact path 'model', registering model based on models:/m-e95396df0f8440e89aa1565980d541f7 instead
2026/03/24 10:05:19 INFO mlflow.store.mode

,experiment_name,experiment_id,run_id,model_family,valid_rmse,valid_r2,model_uri,march_mae,march_rmse,march_r2,registry_name,model_version
0,nyc-green-taxi-gradient-boosting,3,9da3c8600a694c50a00afea247b79b8b,GradientBoostingRegressor,2.614181,0.965949,runs:/9da3c8600a694c50a00afea247b79b8b/model,0.673151,2.276154,0.975985,nyc-green-taxi-gradient-boosting-model,3
1,nyc-green-taxi-random-forest,2,829e3963138a48ec9f37cd702ee7afac,RandomForestRegressor,2.710555,0.963392,runs:/829e3963138a48ec9f37cd702ee7afac/model,0.590735,2.401736,0.973262,nyc-green-taxi-random-forest-model,3
2,nyc-green-taxi-linear-regression,1,468d876f0432457a95c507c89f4d4791,LinearRegression,3.395136,0.942566,runs:/468d876f0432457a95c507c89f4d4791/model,0.825281,2.781218,0.964146,nyc-green-taxi-linear-regression-model,3


## 8. Put the best March model in Production, the other two in Staging

This is done **programmatically**, not in the MLflow UI.

In [31]:
# Identify best model based on March RMSE
best_idx = registered_results_df["march_rmse"].idxmin()
best_row = registered_results_df.loc[best_idx]
best_run_id = best_row["run_id"]

print("\n=== Model Promotion Summary ===")
print("Best model on March (goes to Production):")
print(
    f"{best_row['registry_name']} v{best_row['model_version']} | "
    f"RMSE={best_row['march_rmse']:.4f} | "
    f"R2={best_row['march_r2']:.4f}"
)

print("\nAssigning stages...\n")

# Assign stages
for _, row in registered_results_df.iterrows():
    stage = "Production" if row["run_id"] == best_run_id else "Staging"

    client.transition_model_version_stage(
        name=row["registry_name"],
        version=row["model_version"],
        stage=stage,
        archive_existing_versions=False,
    )

    print(
        f"{row['registry_name']} v{row['model_version']} -> {stage} | "
        f"RMSE={row['march_rmse']:.4f} | "
        f"R2={row['march_r2']:.4f}"
    )


=== Model Promotion Summary ===
Best model on March (goes to Production):
nyc-green-taxi-gradient-boosting-model v3 | RMSE=2.2762 | R2=0.9760

Assigning stages...

nyc-green-taxi-gradient-boosting-model v3 -> Production | RMSE=2.2762 | R2=0.9760
nyc-green-taxi-random-forest-model v3 -> Staging | RMSE=2.4017 | R2=0.9733
nyc-green-taxi-linear-regression-model v3 -> Staging | RMSE=2.7812 | R2=0.9641


## 9. Reproduce the best model from Question 1 (Question 3)

The goal here is simple:

- load the best model back **from MLflow**
- run inference again on the **same validation set** used in Question 1
- compare predictions and metrics

Why use the same validation set?

Because Question 1 selected the best run based on the Jan+Feb validation split, so reproducing it on that exact split is the cleanest check.

In [32]:
best_q1_run_id = overall_best_q1["run_id"]
best_q1_model_uri = overall_best_q1["model_uri"]

print("Best Q1 run_id   :", best_q1_run_id)
print("Best Q1 model_uri:", best_q1_model_uri)

Best Q1 run_id   : 9da3c8600a694c50a00afea247b79b8b
Best Q1 model_uri: runs:/9da3c8600a694c50a00afea247b79b8b/model


In [33]:
# Load the model back from MLflow
reloaded_best_model = mlflow.sklearn.load_model(best_q1_model_uri)

# Run inference again on the SAME validation data from Q1
reloaded_metrics, reloaded_preds = evaluate_regression(reloaded_best_model, X_valid, y_valid)

# For comparison, we also load the original run's metrics from the MLflow run record
original_run = client.get_run(best_q1_run_id)

original_valid_metrics = {
    "mae": float(original_run.data.metrics["valid_mae"]),
    "rmse": float(original_run.data.metrics["valid_rmse"]),
    "r2": float(original_run.data.metrics["valid_r2"]),
}

comparison_df = pd.DataFrame(
    [
        {"metric": "mae", "original_logged": original_valid_metrics["mae"], "reloaded_model": reloaded_metrics["mae"]},
        {"metric": "rmse", "original_logged": original_valid_metrics["rmse"], "reloaded_model": reloaded_metrics["rmse"]},
        {"metric": "r2", "original_logged": original_valid_metrics["r2"], "reloaded_model": reloaded_metrics["r2"]},
    ]
)

comparison_df["absolute_difference"] = (
    comparison_df["original_logged"] - comparison_df["reloaded_model"]
).abs()

comparison_df

,metric,original_logged,reloaded_model,absolute_difference
0,mae,0.646677,0.646677,0.0
1,rmse,2.614181,2.614181,0.0
2,r2,0.965949,0.965949,0.0


In [34]:
# Optional deeper check: compare predictions themselves
# We only have the reloaded predictions now, so let's also load the model one more time
# from the same URI and predict again. This is mainly to show the idea clearly.

same_model_again = mlflow.sklearn.load_model(best_q1_model_uri)
same_preds_again = same_model_again.predict(X_valid)

predictions_exact_match = np.array_equal(reloaded_preds, same_preds_again)
predictions_close_match = np.allclose(reloaded_preds, same_preds_again)

max_abs_prediction_diff = np.max(np.abs(reloaded_preds - same_preds_again))

print("Predictions exact match :", predictions_exact_match)
print("Predictions close match :", predictions_close_match)
print("Max abs prediction diff :", max_abs_prediction_diff)

Predictions exact match : True
Predictions close match : True
Max abs prediction diff : 0.0


## 10. Explanation for Question 3(d)

**Do the reproduced results match?**

They do match **exactly** or are so close that the difference is effectively zero.

Why?

- I loaded the **saved trained model artifact** from MLflow, not a freshly retrained model.
- I used the **same input data**.
- I used the **same preprocessing pipeline** because it was saved inside the model pipeline.